In [4]:
import pandas as pd
import numpy as np

# ============================================================
# Configuration
# ============================================================
COUNTS_FILE    = 'county_radiologist_pathologist_counts.csv'
CENSUS_FILE    = 'co-est2024-alldata.csv'
MORTALITY_FILE = 'death_2023.csv'
INCIDENCE_FILE = 'incd_uscs_2022.csv'
RUCC_FILE      = 'Ruralurbancontinuumcodes2023.csv'
CHR_FILE       = 'analytic_data2023_0.csv'
OUTPUT_FILE    = 'analysis_dataset.csv'

# Connecticut excluded: NCI uses legacy county FIPS (09001-09015),
# Census CO-EST2024 uses new Planning Region FIPS (09110-09190).
# NCI explicitly notes: 'This website still uses Connecticut counties
# instead of planning regions for consistency of geographies across
# data topics.' Geographic unit mismatch precludes valid linkage.
# See also: 87 FR 34235 (Census Bureau, June 6, 2022).
US_STATES = {
    'AL','AK','AZ','AR','CA','CO','DE','DC','FL','GA','HI','ID',
    'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO',
    'MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA',
    'RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
}  # CT intentionally excluded

state_abbrev = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Delaware':'DE','District of Columbia':'DC',
    'Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID','Illinois':'IL',
    'Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA',
    'Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY',
    'North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR',
    'Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD',
    'Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA',
    'Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY'
}

# ============================================================
# Step 1: Load provider counts
# ============================================================
print("Loading provider counts...")
counts = pd.read_csv(COUNTS_FILE, dtype=str)

for col in ['count_radiologist', 'count_radiologist_primary', 'count_pathologist']:
    counts[col] = pd.to_numeric(counts[col], errors='coerce').fillna(0)

print(f"Provider counts loaded: {len(counts):,} counties")

# ============================================================
# Step 2: Load Census population data
# ============================================================
print("\nLoading Census population data...")
census = pd.read_csv(CENSUS_FILE, encoding='latin-1', low_memory=False,
                     usecols=['SUMLEV','STATE','COUNTY','STNAME','CTYNAME',
                              'POPESTIMATE2021'])

census = census[census['SUMLEV'] == 50].copy()
census['FIPS'] = (census['STATE'].astype(str).str.zfill(2) +
                  census['COUNTY'].astype(str).str.zfill(3))
census['state'] = census['STNAME'].map(state_abbrev)
census = census[census['state'].isin(US_STATES)].copy()
census = census.rename(columns={
    'POPESTIMATE2021': 'population_2021',
    'CTYNAME':         'county_name',
    'STNAME':          'state_name'
})

print(f"Census counties loaded: {len(census):,} (CT excluded)")

# ============================================================
# Step 3: Merge counts with Census
# ============================================================
print("\nMerging provider counts with Census...")
df = census[['FIPS','state','state_name','county_name','population_2021']].merge(
    counts[['FIPS','count_radiologist','count_radiologist_primary','count_pathologist']],
    on='FIPS',
    how='left'
)

for col in ['count_radiologist','count_radiologist_primary','count_pathologist']:
    df[col] = df[col].fillna(0)

print(f"After merge: {len(df):,} counties")
print(f"Counties with zero radiologists: {(df['count_radiologist'] == 0).sum():,}")

# ============================================================
# Step 4: Calculate density (per 100,000 population)
# ============================================================
print("\nCalculating density per 100,000 population...")
df['population_2021'] = pd.to_numeric(df['population_2021'], errors='coerce')

df['density_radiologist']         = df['count_radiologist']         / df['population_2021'] * 100_000
df['density_radiologist_primary'] = df['count_radiologist_primary'] / df['population_2021'] * 100_000
df['density_pathologist']         = df['count_pathologist']         / df['population_2021'] * 100_000

for col in ['density_radiologist','density_radiologist_primary','density_pathologist']:
    df[col] = df[col].round(4)

# ============================================================
# Step 5: Load USDA Rural-Urban Continuum Codes (1-9 numeric)
# Source: USDA ERS, 2023 Rural-Urban Continuum Codes
# ============================================================
print("\nLoading USDA Rural-Urban Continuum Codes (2023)...")
rucc_raw = pd.read_csv(RUCC_FILE, dtype=str, encoding='latin-1')

rucc = rucc_raw[rucc_raw['Attribute'] == 'RUCC_2023'][['FIPS','Value']].copy()
rucc = rucc.rename(columns={'Value': 'rucc_2023'})
rucc['rucc_2023'] = pd.to_numeric(rucc['rucc_2023'], errors='coerce')
rucc['FIPS'] = rucc['FIPS'].astype(str).str.zfill(5)

print(f"RUCC records: {len(rucc):,}")
df = df.merge(rucc[['FIPS','rucc_2023']], on='FIPS', how='left')
print(f"Counties missing RUCC: {df['rucc_2023'].isna().sum()}")

# ============================================================
# Step 6: Load County Health Rankings — income and uninsured
# Source: County Health Rankings 2023 (analytic_data2023_0.csv)
# v063_rawvalue = Median Household Income (USD)
# v003_rawvalue = Uninsured Adults (proportion 0-1)
# ============================================================
print("\nLoading County Health Rankings (income + uninsured)...")
chr_raw = pd.read_csv(CHR_FILE, skiprows=1, low_memory=False,
                      usecols=['fipscode','v063_rawvalue','v003_rawvalue'])

# County level only: 5-digit FIPS, not state (ends in 000) or national (00000)
chr_raw['fipscode'] = chr_raw['fipscode'].astype(str).str.zfill(5)
chr_county = chr_raw[
    chr_raw['fipscode'].str.match(r'^\d{5}$', na=False) &
    (chr_raw['fipscode'] != '00000') &
    (~chr_raw['fipscode'].str.endswith('000'))
].copy()

chr_county = chr_county.rename(columns={
    'fipscode':       'FIPS',
    'v063_rawvalue':  'median_household_income',
    'v003_rawvalue':  'pct_uninsured_adults'
})

chr_county['median_household_income'] = pd.to_numeric(
    chr_county['median_household_income'], errors='coerce')
# Convert proportion to percentage (0.12 → 12.0)
chr_county['pct_uninsured_adults'] = pd.to_numeric(
    chr_county['pct_uninsured_adults'], errors='coerce') * 100

print(f"CHR county records: {len(chr_county):,}")
df = df.merge(chr_county[['FIPS','median_household_income','pct_uninsured_adults']],
              on='FIPS', how='left')

chr_missing = df['median_household_income'].isna().sum()
print(f"Counties missing CHR income/uninsured: {chr_missing}")

# ============================================================
# Step 7: Load NCI Mortality data
# ============================================================
print("\nLoading NCI mortality data...")
mort = pd.read_csv(MORTALITY_FILE, skiprows=8, dtype=str)
mort = mort[mort['FIPS'] != '00000'].copy()
mort['FIPS'] = mort['FIPS'].astype(str).str.zfill(5)

mort = mort.rename(columns={
    'Age-Adjusted Death Rate([rate note]) - deaths per 100,000': 'mortality_rate',
    'Recent Trend':         'mortality_recent_trend',
    'Average Annual Count': 'mortality_avg_annual_count'
})

mort = mort[['FIPS','mortality_rate',
             'mortality_recent_trend','mortality_avg_annual_count']].copy()

mort['mortality_rate_suppressed'] = mort['mortality_rate'].str.strip() == '*'
mort['mortality_rate'] = pd.to_numeric(
    mort['mortality_rate'].str.strip().replace('*', np.nan), errors='coerce'
)

print(f"Mortality records: {len(mort):,}")
print(f"Suppressed mortality rates: {mort['mortality_rate_suppressed'].sum():,}")

# ============================================================
# Step 8: Load NCI Incidence data
# ============================================================
print("\nLoading NCI incidence data...")
incd = pd.read_csv(INCIDENCE_FILE, skiprows=8, dtype=str)
incd = incd[incd['FIPS'] != '00000'].copy()
incd['FIPS'] = incd['FIPS'].astype(str).str.zfill(5)

incd = incd.rename(columns={
    'Age-Adjusted Incidence Rate([rate note]) - cases per 100,000': 'incidence_rate',
    'Recent Trend':         'incidence_recent_trend',
    'Average Annual Count': 'incidence_avg_annual_count'
})

incd = incd[['FIPS','incidence_rate',
             'incidence_recent_trend','incidence_avg_annual_count']].copy()

incd['incidence_rate_suppressed'] = incd['incidence_rate'].str.strip() == '*'
incd['incidence_rate'] = pd.to_numeric(
    incd['incidence_rate'].str.strip().replace('*', np.nan), errors='coerce'
)

print(f"Incidence records: {len(incd):,}")
print(f"Suppressed incidence rates: {incd['incidence_rate_suppressed'].sum():,}")

# ============================================================
# Step 9: Final merge + CT exclusion safety net
# ============================================================
print("\nMerging all datasets...")
df = df.merge(mort, on='FIPS', how='left')
df = df.merge(incd, on='FIPS', how='left')

# Explicit CT filter as safety net (already excluded via US_STATES above)
df = df[df['state'] != 'CT'].copy()

print(f"Final dataset: {len(df):,} counties (CT excluded)")

# ============================================================
# Step 10: Summary + Suppressed / Zero-provider overlap check
# ============================================================
print("\n=== Final Dataset Summary ===")
print(f"Total counties:                     {len(df):,}")
print(f"Counties with mortality data:       {df['mortality_rate'].notna().sum():,}")
print(f"Counties with incidence data:       {df['incidence_rate'].notna().sum():,}")
print(f"Counties with RUCC data:            {df['rucc_2023'].notna().sum():,}")
print(f"Counties with income data:          {df['median_household_income'].notna().sum():,}")
print(f"Counties with uninsured data:       {df['pct_uninsured_adults'].notna().sum():,}")
print(f"Counties with suppressed mortality: {df['mortality_rate_suppressed'].sum():,}")
print(f"Counties with suppressed incidence: {df['incidence_rate_suppressed'].sum():,}")

print("\n=== Suppressed Outcome / Zero-Provider Overlap ===")
zero_rad     = df['count_radiologist'] == 0
supp_mort    = df['mortality_rate_suppressed'] == True
supp_incd    = df['incidence_rate_suppressed'] == True
overlap_mort = (zero_rad & supp_mort).sum()
overlap_incd = (zero_rad & supp_incd).sum()

print(f"Zero-radiologist counties:                  {zero_rad.sum():,}")
print(f"Suppressed mortality counties:              {supp_mort.sum():,}")
print(f"Suppressed incidence counties:              {supp_incd.sum():,}")
print(f"Overlap: zero-rad AND suppressed mortality: {overlap_mort:,} "
      f"({overlap_mort / zero_rad.sum() * 100:.1f}% of zero-rad counties)")
print(f"Overlap: zero-rad AND suppressed incidence: {overlap_incd:,} "
      f"({overlap_incd / zero_rad.sum() * 100:.1f}% of zero-rad counties)")
print("\nNote: These counties are systematically excluded from regression.")
print("Document in Limitations section.")

print("\n=== Control Variable Distributions ===")
print(f"RUCC 2023: {df['rucc_2023'].value_counts().sort_index().to_dict()}")
print(f"\nMedian Household Income (USD):")
print(df['median_household_income'].describe().round(0))
print(f"\nUninsured Adults (%):")
print(df['pct_uninsured_adults'].describe().round(2))

print("\n=== Radiologist Density Distribution ===")
print(df['density_radiologist'].describe().round(4))

print("\n=== Mortality Rate Distribution ===")
print(df['mortality_rate'].describe().round(2))

# ============================================================
# Step 11: Save
# ============================================================
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved to: {OUTPUT_FILE}")
print(f"Columns: {df.columns.tolist()}")
print("\nNext step: run 4__ols_regression.py")

Loading provider counts...
Provider counts loaded: 3,144 counties

Loading Census population data...
Census counties loaded: 3,135 (CT excluded)

Merging provider counts with Census...
After merge: 3,135 counties
Counties with zero radiologists: 1,250

Calculating density per 100,000 population...

Loading USDA Rural-Urban Continuum Codes (2023)...
RUCC records: 3,233
Counties missing RUCC: 0

Loading County Health Rankings (income + uninsured)...
CHR county records: 3,142
Counties missing CHR income/uninsured: 3

Loading NCI mortality data...
Mortality records: 3,159
Suppressed mortality rates: 59

Loading NCI incidence data...
Incidence records: 3,160
Suppressed incidence rates: 9

Merging all datasets...
Final dataset: 3,135 counties (CT excluded)

=== Final Dataset Summary ===
Total counties:                     3,135
Counties with mortality data:       3,073
Counties with incidence data:       3,018
Counties with RUCC data:            3,135
Counties with income data:          3,13